# internsg.com Search

## Production Code
##### This code runs in a virtual environment using Visual Studio Code.

In [2]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

def create_chrome_driver(headless=True):
    # Set up Chrome options
    chrome_options = Options()
    if headless: # Running in the headless mode will be faster
        chrome_options.add_argument("--headless")

    # Other options
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920x1080")

    # Return the Chrome Driver
    return webdriver.Chrome(options=chrome_options)

def extract_contact_info(detail_soup):
    # Check for the 'Application Instructions' section
    instructions_section = detail_soup.find('div', class_='ast-col-md-2 font-weight-bold', string='Application Instructions')

    global contact_links
    email_present = False
    email = None

    contact_links = []
    if instructions_section:
        instructions_div = instructions_section.find_next_sibling('div')
        if instructions_div:
            # Check if 'data-enc-email' is mentioned in the 'Application Instructions' section
            if 'data-enc-email' in instructions_div.decode_contents().lower():
                email_present = True

            # Find all the links provided in the 'Application Instructions' section
            for link in instructions_div.find_all('a'):
                contact_links.append(link.get('href'))

            # Find the email address provided in the 'Application Instructions' section
            email_span = instructions_div.find('span')
            if email_span:
                email = email_span.text.strip()
                print("Extracted Email: ", email)
            else:
                print("No email span found.")
        else:
            print("No instructions div found after the section.")

    print('Email:', email)
    print('Email present:', email_present)
    return email, email_present

def extract_company_name(detail_soup):
    company_name = None
    domain = None

    # Extract the email domain in the 'Company' section
    company_section = detail_soup.find('div', class_='ast-col-md-2 font-weight-bold', string='Company')
    if company_section:
        company_div = company_section.find_next_sibling('div')
        if company_div:
            domain_span = company_div.find('span')
            if domain_span:
                domain = domain_span.text.strip()
            else:
                print("No domain span found.")
        else:
            print("No company div found after the company section.")
    else:
        print("No company section found.")

    # Extract the company name
    title_tag = detail_soup.find('title')
    if title_tag:
        title_text = title_tag.get_text(strip=True)
        if ' - ' in title_text:
            company_name = title_text.split(' - ')[0]
        else:
            print('No - found.')
    else:
        company_name = 'Unknown'
        print("No title tag found.")

    print('Company name:', company_name)
    print('Domain:', domain)
    return company_name, domain

def search_for_email(company_name, domain):
    email_addresses = set()
    if '&' in company_name: # The search_query cannot contain '&'
        company_name = company_name.replace('&', ' ')

    try: # Search email addresses on Google
        search_query = f"{company_name} {domain} contact email"
        driver = create_chrome_driver(headless=True)
        driver.get(f"https://www.google.com/search?q={search_query}")
        time.sleep(4)  # Wait for the page to load

        # Grab the page text of search results
        page_text = driver.find_element(By.TAG_NAME, 'body').text
        # Extract emails from the search results page text
        email_regex = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
        found_emails = re.findall(email_regex, page_text)
        # Check the domain and add found emails to the set
        for email in found_emails:
            if domain in email: # Check if the email belongs to the correct domain
                email_addresses.add(email)
                print(f"Found email from search results page text: {email}")

        # Grab search results
        search_results = driver.find_elements(By.CSS_SELECTOR, 'h3')
        contact_found = False
        # Filter links that contain "Contact", "About", or "Privacy"
        for result in search_results:
            link = result.find_element(By.XPATH, '..').get_attribute('href')
            title = result.text
            if ('contact' or 'about' or 'privacy') in title.lower():
                contact_links.append(link)
                print(f"Found contact link: {link}")
                contact_found = True

        if not contact_found:
            for result in search_results[:3]: # Select first three results if no contact found
                link = result.find_element(By.XPATH, '..').get_attribute('href')
                contact_links.append(link)
                print(f"Added link from first three results: {link}")

        # Visit each contact link
        for link in contact_links:
            print(f"Visiting {link}")
            driver.get(link)
            time.sleep(4)  # Wait for the page to load

            # Search for any email addresses on the page
            page_text = driver.find_element(By.TAG_NAME, 'body').text
            email_regex = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
            found_emails = re.findall(email_regex, page_text)

            # Check the domain and add found emails to the set
            for email in found_emails:
                if domain in email: # Check if the email belongs to the correct domain
                    email_addresses.add(email)
                    print(f"Found email from body text: {email}")

            driver.back() # Go back to Google results
            time.sleep(0.5)  # Wait before visiting the next link

    except Exception as e:
        print("Page load failed:", str(e))

    finally:
        driver.quit()

    return email_addresses

def choose_best_email(email_addresses_list):
    # Given a list of email addresses, return the most relevant one based on predefined keywords
    # Define keywords related to jobs (the first keyword is the most relevant)
    keywords = ['careers', 'career', 'jobs', 'job', 'hr', 'contact', 'info', 'admin', 'customerservice',
                'hello', 'hi', 'support', 'help', 'sales', 'marketing']

    # If the list has only one email, return it directly
    if len(email_addresses_list) == 1:
        return email_addresses_list[0]

    for keyword in keywords:
        for email in email_addresses_list:
            email_username = email.split('@')[0].lower()
            # If the keyword is found in the username, return that email address
            if keyword in email_username:
                return email

    return None # Return None if no relevant email is found

def save_to_excel(data, filename='Joblist updated to 2025.1.XX.xlsx'):
    # Check if the file already exists
    file_exists = os.path.isfile(filename)

    if not file_exists:
        # Create a new file with headers if it doesn't exist
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df = pd.DataFrame(data, columns=['Company', 'Position', 'E-mail', 'E-mail List', 'Domain', 'Date', 'Link'])
            df.to_excel(writer, index=False, sheet_name='Sheet1')
        print(f"Initial data written to {filename}")
    else:
        # Append to the existing file
        with pd.ExcelWriter(filename, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
            df = pd.DataFrame(data, columns=['Company', 'Position', 'E-mail', 'E-mail List', 'Domain', 'Date', 'Link'])
            startrow = writer.sheets['Sheet1'].max_row  # Get the last row
            df.to_excel(writer, index=False, header=False, startrow=startrow, sheet_name='Sheet1')
        print(f"Data appended to {filename}")

def process_batch(start_page, end_page):
    batch_data = []
    for page_num in range(start_page, end_page + 1):
        url = f'https://www.internsg.com/jobs/{page_num}/?f_p&f_i&filter_s#isg-top'
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find all internship listings on the page
        listings = soup.find_all('div', class_='ast-row')

        for listing in listings:
            # Ensure the company div exists before accessing it
            company_div = listing.find('div', class_='ast-col-lg-3')
            if not company_div:
                continue  # Skip if company div is not found

            position_div = company_div.find_next_sibling('div')
            if not position_div:
                continue  # Skip if position div is not found

            # Extract the link to the detailed internship page
            detail_link_tag = position_div.find('a')
            if not detail_link_tag:
                continue  # Skip if detail link is not found

            detail_link = detail_link_tag['href']

            # Extract the post status
            post_status_div = listing.find('div', class_='ast-col-lg-1')
            if not post_status_div:
                continue  # Skip if post status div is not found

            post_status = post_status_div.get_text(strip=True)

            # Only proceed if the post is not closed
            if post_status.lower() != 'closed':
                position_title = position_div.get_text(strip=True)
                if 'intern' in position_title.lower(): # Check if title contains 'intern'
                    print('\nPosition title:', position_title)
                    date_span = post_status_div.find('span')
                    if date_span:
                        date = date_span.get_text(strip=True) # Get the post date

                    while True:
                        # Visit the detailed internship page
                        driver = create_chrome_driver(headless=True)
                        driver.get(detail_link)
                        html = driver.page_source
                        detail_soup = BeautifulSoup(html, 'html.parser')
                        driver.quit()

                        # Loop until the company name and the domain can be extracted
                        company_name, domain = extract_company_name(detail_soup)
                        if (company_name and domain is not None):
                            break
                        else:
                            time.sleep(0.5)

                    # Extract email
                    best_email = None
                    email_addresses_list = []
                    email, email_present = extract_contact_info(detail_soup)
                    if not email_present:
                        # If no email is found in the 'Application Instructions' section, search it on Google
                        email_addresses = search_for_email(company_name, domain)
                        email_addresses_list = list(email_addresses)
                        print(email_addresses_list)
                        if len(email_addresses_list) >= 1:
                            # If emails are found on Google, go to select the most appropriate email address
                            best_email = choose_best_email(email_addresses_list)
                            print(f"The most appropriate email address is: {best_email}")
                    else:
                        best_email = email

                    # Append the extracted data to the list
                    record = [
                        company_name,
                        position_title,
                        best_email,
                        email_addresses_list,
                        domain,
                        date,
                        detail_link
                    ]
                    save_to_excel([record]) # Save the data to the Excel file
                    print('Record saved')

        # Print status and cumulative time
        elapsed_time = time.time() - start_time
        print(f"Finished processing main listing page {page_num}")
        print(f"Cumulative running time: {elapsed_time:.2f} seconds")

    return batch_data

# Initialize an empty list to store the internship details
internship_data = []

# Measure the start time
start_time = time.time()

# Process the pages (you can adjust the range as needed)
batch_data = process_batch(66, 67)

# Measure the end time
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total running time: {elapsed_time:.2f} seconds")



Position title: Operations & Marketing Intern
Company name: Sojourner: Journeys of Inspiration
Domain: sojourner.sg
Extracted Email:  david@sojourner.sg
Email: david@sojourner.sg
Email present: True
Initial data written to Joblist updated to 2025.1.XX.xlsx
Record saved

Position title: Strategy & Operations Intern
Company name: Urban Company
Domain: urbancompany.com
No email span found.
Email: None
Email present: False
Found email from search results page text: support@urbancompany.com
Found email from search results page text: press@urbancompany.com
Found email from search results page text: janedoe@urbancompany.com
Found email from search results page text: help@urbancompany.com
Added link from first three results: https://www.urbancompany.com/contact-us
Added link from first three results: https://kanoon360.com/blog/urban-company-customer-care-number/
Added link from first three results: https://www.urbancompany.com/helpcenter
Visiting https://www.urbancompany.com/contact-us
Found 

## Optional
##### You may choose to simply delete all the records that contain nothing in the E-mail column by running this code cell.

In [3]:
def delete_empty_emails(file_path):
    df = pd.read_excel(file_path)
    df = df[df['E-mail'].notnull()]
    df.to_excel(file_path, index=False)

delete_empty_emails('Joblist updated to 2025.1.XX.xlsx')